# 前置章节复习总结（3.0-4.7）

目标：
- 快速回忆 3.0 到 4.7 的核心内容（排除 4.8 本体）
- 用“直觉 + 公式 + 最小代码”方式复盘
- 直接迁移到 4.8 Kaggle 房价预测实战

适合：2个月后回看、想迅速恢复手感的初学者

## 0. 一张图看全局（学习主线）

1. 数据预处理（3.0）：把脏数据变成可训练输入。
2. 线性回归（3.1）：建立回归任务最小可行 baseline。
3. Softmax（3.4）：理解分类概率输出和损失设计。
4. 多层感知机（4.0）：引入非线性，提高拟合能力。
5. 模型选择（4.4）：用训练/验证曲线识别欠拟合和过拟合。
6. 权重衰退（4.5）：抑制参数过大，提升泛化。
7. Dropout（4.6）：训练时随机屏蔽神经元，降低共适应。
8. 数值稳定性与初始化（4.7）：让训练过程更稳、更快收敛。

## 1. 3.0 数据预处理：模型前的地基

直觉作用：
- 模型像厨师，原始数据像食材；不清洗直接下锅，结果一定不稳定。

关键公式：

标准化（z-score）：
$$
x' = \frac{x - \mu}{\sigma}
$$

缺失值填补（均值法）：
$$
x_{miss} \leftarrow \bar{x}
$$

什么时候用：
- 数值特征量纲差异大（如面积 vs 房间数）
- 存在缺失值、类别特征

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'LotArea': [8450, 9600, None, 9550],
    'Neighborhood': ['A', 'B', 'A', None]
})

# 1) 数值缺失值填充
num_cols = ['LotArea']
df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

# 2) 类别缺失值填充后 one-hot
cat_cols = ['Neighborhood']
df[cat_cols] = df[cat_cols].fillna('Unknown')
df_encoded = pd.get_dummies(df, columns=cat_cols, dummy_na=False)

# 3) 数值标准化
df_encoded['LotArea'] = (df_encoded['LotArea'] - df_encoded['LotArea'].mean()) / df_encoded['LotArea'].std()

df_encoded

## 2. 3.1 线性回归：回归任务 baseline

直觉作用：
- 先用最简单模型建立参照线，后续模型都和它比。

关键公式：
$$
\hat{y} = \mathbf{w}^\top \mathbf{x} + b
$$

均方误差：
$$
\mathcal{L}=\frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i-y_i)^2
$$

什么时候用：
- 先快速验证特征是否有预测信号

In [ ]:
import torch

X = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[3.0], [5.0], [7.0], [9.0]])  # 近似 y=2x+1

model = torch.nn.Linear(1, 1)
loss_fn = torch.nn.MSELoss()
optim = torch.optim.SGD(model.parameters(), lr=0.1)

for _ in range(100):
    pred = model(X)
    loss = loss_fn(pred, y)
    optim.zero_grad()
    loss.backward()
    optim.step()

print('w,b =', model.weight.item(), model.bias.item())

## 3. 3.4 Softmax 回归：分类任务概率输出

直觉作用：
- 把每个类别的“打分”变成概率分布，适合多分类。

关键公式：
$$
\text{softmax}(z_i)=\frac{e^{z_i}}{\sum_j e^{z_j}}
$$

交叉熵（单样本）：
$$
\mathcal{L}=-\sum_k y_k\log \hat{p}_k
$$

什么时候用：
- 标签是离散类别（本章用于理解分类，房价本身仍是回归）

In [ ]:
logits = torch.tensor([[2.0, 1.0, 0.1]])
probs = torch.softmax(logits, dim=1)
print('分类概率:', probs)
print('预测类别:', torch.argmax(probs, dim=1).item())

## 4. 4.0 多层感知机（MLP）：给模型非线性表达力

直觉作用：
- 线性回归像画直线；MLP 像画弯曲曲线，可以拟合更复杂关系。

关键公式：
$$
\mathbf{h}=\sigma(W_1\mathbf{x}+b_1),\quad \hat{y}=W_2\mathbf{h}+b_2
$$

什么时候用：
- baseline 欠拟合明显，且数据量、特征信息足够

In [ ]:
mlp = torch.nn.Sequential(
    torch.nn.Linear(10, 32),
    torch.nn.ReLU(),
    torch.nn.Linear(32, 1)
)

x_demo = torch.randn(4, 10)
out_demo = mlp(x_demo)
print('MLP 输出形状:', out_demo.shape)

## 5. 4.4 模型选择：识别欠拟合/过拟合

直觉作用：
- 训练集好不代表真的好，验证集才是“模拟考试”。

关键判断：
- 欠拟合：训练误差高，验证误差也高。
- 过拟合：训练误差低，但验证误差高。

常见指标（回归）：
$$
RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^n(\hat{y}_i-y_i)^2}
$$

In [ ]:
# 伪造一组训练/验证损失，演示如何看趋势
train_loss = [1.2, 0.8, 0.5, 0.35, 0.25]
val_loss = [1.3, 0.9, 0.7, 0.72, 0.85]

print('训练损失:', train_loss)
print('验证损失:', val_loss)
print('现象: 训练继续下降但验证后期上升 -> 过拟合信号')

## 6. 4.5 权重衰退（L2）：防止模型记死训练集

直觉作用：
- 给大权重“收税”，让模型别走极端。

关键公式：
$$
\mathcal{L}_{total}=\mathcal{L}_{data}+\lambda\|\mathbf{w}\|_2^2
$$

什么时候用：
- 特征多、样本相对少，验证误差明显高于训练误差

In [ ]:
linear = torch.nn.Linear(10, 1)
optimizer_wd = torch.optim.Adam(linear.parameters(), lr=1e-2, weight_decay=1e-3)
print('已在优化器中启用 weight_decay=1e-3')

## 7. 4.6 Dropout：训练时随机断开，提升泛化

直觉作用：
- 类似“每次训练让不同小组成员上场”，降低神经元之间的过度依赖。

关键公式（训练期）：
$$
\tilde{h}=m\odot h,\quad m\sim Bernoulli(1-p)
$$

什么时候用：
- 深层网络过拟合明显时；回归任务通常用较小 dropout（如 0.1~0.3）

In [ ]:
net = torch.nn.Sequential(
    torch.nn.Linear(10, 64),
    torch.nn.ReLU(),
    torch.nn.Dropout(p=0.2),
    torch.nn.Linear(64, 1)
)
print(net)

## 8. 4.7 数值稳定性、初始化、激活函数

直觉作用：
- 训练不稳定往往不是数据错，而是“梯度传不动”或“梯度太大”。

关键点：
- ReLU 常用于深层网络，减缓梯度消失。
- Xavier/Kaiming 初始化可让前期梯度更稳定。

简化理解：
- 初始化太小：信号变弱，学得慢。
- 初始化太大：信号爆炸，loss 震荡甚至 NaN。

In [ ]:
layer = torch.nn.Linear(20, 10)
torch.nn.init.kaiming_uniform_(layer.weight, nonlinearity='relu')
print('初始化后权重均值:', layer.weight.mean().item())
print('初始化后权重标准差:', layer.weight.std().item())

## 9. 迁移到 4.8 房价预测：直接可用清单

### 9.1 可直接复用
1. 数值特征统一标准化，类别特征 one-hot。
2. 训练集与测试集使用同一套预处理逻辑。
3. 先跑线性回归 baseline，再尝试 MLP。
4. 用验证集或 K 折评估，不只看训练误差。

### 9.2 训练与验证策略
- 指标优先 RMSE（回归任务直观）。
- 小数据集建议 K 折交叉验证（如 K=5）。
- 发现过拟合就加正则化、减小模型复杂度或早停。

### 9.3 超参数起点（可先照抄）
- `lr`: 1e-2（Adam）
- `weight_decay`: 1e-4 或 1e-3
- `dropout`: 0.1 到 0.3
- `batch_size`: 64 或 128
- `epochs`: 100 起，结合验证曲线早停

## 10. 30 分钟复习计划 + 自测

复习顺序（30分钟）：
1. 0-8 分钟：复盘 3.0 数据预处理，确保能口述处理流程。
2. 8-15 分钟：复盘 3.1 + 4.0，明确线性与MLP差异。
3. 15-23 分钟：复盘 4.4，练习看训练/验证曲线。
4. 23-30 分钟：复盘 4.5/4.6/4.7，整理调参顺序。

我已经掌握（自测3题）：
- 我能解释为什么房价任务要做标准化。
- 我能说出过拟合和欠拟合在曲线上的区别。
- 我知道 `weight_decay` 的作用是什么。

我还不稳（自测3题）：
- Dropout 在训练和推理阶段的行为差异是什么？
- 当 loss 震荡时我先改学习率还是先改初始化？
- 为什么要先做 baseline 再加复杂网络？